# Count Models: Geometric, Hypergeometric, And Poisson

**Official MA1001B Alignment:** *2.4 geometric and negative binomial; 2.5 hypergeometric; 2.6 Poisson; 2.7 data science links.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Model event occurrences over fixed intervals using the Poisson probability distribution.
- Check fundamental Poisson model assumptions by evaluating the variance-to-mean ratio (dispersion).
- Identify real-world overdispersion caused by time-varying rates and heterogeneous conditions.
- Use Poisson tail probabilities to evaluate infrastructure capacity and peak demand risks.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model arrival counts per unit time while testing whether arrival rates remain constant across time intervals.
- **2. Computational Link (How Python represents it):** We compute empirical variance-to-mean ratios in Pandas and evaluate Poisson survival functions (`stats.poisson.sf`) in SciPy.
- **3. Decision Link (How it guides action):** Detecting overdispersion prevents underestimating peak demand surges, guiding safer capacity and inventory planning.


## Decision Scenario

> **The Problem:** A mobility planner wants to model hourly bike demand. A simple count model may help, but only if its assumptions are reasonable.


## Conceptual Explanation

Count models describe nonnegative integer outcomes. A Poisson model is useful for counts in a fixed interval when events occur independently at a roughly constant rate. Real data often violate that constant-rate assumption because time, weather, and context change demand.


## Mathematical Anchor

If X follows Poisson(lambda), then P(X = k) = exp(-lambda) lambda^k / k!, and E[X] = Var(X) = lambda.


## Data And Workflow Notes

Uses Bike Sharing Demand if available; otherwise simulates counts with daily variation.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Data Acquisition & Hourly Count Setup

We load the Kaggle Bike Sharing Demand dataset if available; otherwise, we simulate hourly bike counts incorporating morning and evening rush-hour peaks.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Load Bike Sharing Demand data or simulate time-varying hourly counts
path = Path("data/raw/bike-sharing/train.csv")
if path.exists():
    bike = pd.read_csv(path)
    counts = bike["count"].dropna()
else:
    print(f"Notice: Missing {path}. Using fallback simulation with rush-hour peaks.")
    hour = np.tile(np.arange(24), 60)
    # Rate varies significantly by hour: baseline 40, morning rush +90, evening rush +80
    rate = 40 + 90 * ((hour >= 7) & (hour <= 9)) + 80 * ((hour >= 17) & (hour <= 19))
    counts = pd.Series(rng.poisson(rate), name="hourly_bike_count")

counts.describe().round(2)


### Step 2: Checking the Poisson Dispersion Assumption

A theoretical Poisson distribution requires variance equal to mean (ratio = 1.0). We compute the empirical variance-to-mean ratio to check for overdispersion.


In [ ]:
# Check Poisson assumption: Var(X) / E[X] should be approximately 1.0
mean_val = counts.mean()
var_val = counts.var(ddof=1)

pd.Series({
    "empirical_mean": mean_val,
    "empirical_variance": var_val,
    "variance_to_mean_ratio": var_val / mean_val,
    "is_overdispersed": (var_val / mean_val) > 1.5
}).round(2)


### Step 3: Visualizing Observed Count Distributions

We plot the histogram of hourly bike counts to inspect skewness, multi-modality, and extreme peak demand hours.


In [ ]:
# Plot histogram of hourly demand counts
ax = sns.histplot(counts, bins=30, kde=True, color="seagreen")
ax.set_title("Observed Hourly Bike Demand Distribution", fontsize=14, pad=10)
ax.set_xlabel("Hourly Bike Rentals", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
plt.show()


### Step 4: Evaluating Capacity Risk via Tail Probabilities

We estimate the probability of exceeding the 90th percentile demand threshold under both the empirical data and a naive single-rate Poisson model.


In [ ]:
# Compare empirical peak risk against naive Poisson model predictions
lambda_hat = counts.mean()
threshold = counts.quantile(0.90)

pd.Series({
    "estimated_lambda": lambda_hat,
    "90th_percentile_capacity_threshold": threshold,
    "observed_P(demand >= threshold)": (counts >= threshold).mean(),
    "naive_poisson_P(demand >= threshold)": stats.poisson.sf(threshold - 1, lambda_hat),
}).round(4)


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Use the variance-to-mean ratio to decide whether a single Poisson model is plausible.

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Assuming all count data automatically follow a Poisson distribution without checking dispersion.
- **Warning:** Ignoring time segmentation when the underlying event rate clearly changes across hours or seasons.
- **Warning:** Comparing only averages across groups while ignoring massive differences in variance and extreme peaks.


## Independent Practice

> [!TIP]
> **Your Task:**
> Create two time segments, such as high-demand rush hours and low-demand off-peak hours. Compare the mean, variance, and variance-to-mean ratio for each segment independently.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What does 'overdispersion' mean in practical operational planning language?

*Write your brief conceptual reflection below:*
